In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime 

CATALOG = dbutils.widgets.get("catalog")
RAW_SCHEMA = dbutils.widgets.get("stream_schema")
TITLE = dbutils.widgets.get("title")

BASE_PATH = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/raw/lab4_streaming"

SILVER_SCHEMA = dbutils.widgets.get("silver_schema")
SCD2_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.{TITLE}_scd2"
SCD2_CHECKPOINT = f"{BASE_PATH}/checkpoints/silver_scd2"

In [0]:
%run ./03_silver_cleaning

In [0]:
def merge_scd2(micro_batch_df, batch_id):
    print(f"Processing batch: {batch_id}")

    dedup_window = (
        Window
        .partitionBy("show_id")
        .orderBy(F.col("ingestion_time").desc())
    )

    deduplicated_df = (
        micro_batch_df
        .withColumn("_row_number", F.row_number().over(dedup_window))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
        .withColumn("valid_from", F.current_timestamp())
        .withColumn("valid_to", F.lit(None).cast("timestamp"))
        .withColumn("is_current", F.lit(True))
    )

    if not spark.catalog.tableExists(SCD2_TABLE):
        (
            deduplicated_df.limit(0).write
                .format("delta")
                .saveAsTable(SCD2_TABLE)
        )
    
    target = DeltaTable.forName(spark, SCD2_TABLE)

    current_target_df = (
        spark.table(SCD2_TABLE).filter(F.col("is_current") == True)
    )

    business_columns = [
        col for col in deduplicated_df.columns if col not in ["valid_from", "valid_to", "is_current", "ingestion_time", "silver_created_at", "silver_updated_at"]
    ]

    change_condition = None 
    for col_name in business_columns:
        condition = ~F.col(f"source.{col_name}").eqNullSafe(F.col(f"target.{col_name}"))
        if change_condition is None:
            change_condition = condition
        else:
            change_condition = change_condition | condition
    
    comparison_df = (
        deduplicated_df.alias("source").join(
            current_target_df.alias("target"),
            F.col("source.show_id") == F.col("target.show_id"), "left"
        )
    )

    changed_df = (
        comparison_df.filter(
            F.col("target.show_id").isNotNull() & change_condition 
        )
        .select("source.*")
    )

    new_df = (
        comparison_df.filter(F.col("target.show_id").isNull())
        .select("source.*")
    )

    (target.alias("target")
        .merge(
            changed_df.alias("source"),
            """
            target.show_id = source.show_id
            AND target.is_current = true
            """
        )
        .whenMatchedUpdate(
            set={
                "is_current": "false",
                "valid_to": "current_timestamp()"
            }
        )
        .execute()
    )

    records_to_insert = new_df.unionByName(changed_df)

    if not records_to_insert.isEmpty():
        (
            records_to_insert.write
            .format("delta")
            .mode("append")
            .saveAsTable(SCD2_TABLE)
        )


In [0]:
query = (
    silver_df.writeStream
        .foreachBatch(merge_scd2)
        .option("checkpointLocation", SCD2_CHECKPOINT)
        .trigger(availableNow=True)
        .start()
)

query.awaitTermination()